## Chains

In [1]:
from dotenv import load_dotenv, find_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq


# Load environment variables from .env
_ = load_dotenv(find_dotenv())

# Create a ChatOpenAI model
# model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.7)

# Define prompt templates (no need for separate Runnable chains)
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a comedian who tells jokes about {topic}."),
        ("human", "Tell me {joke_count} jokes."),
    ]
)

# Create the combined chain using LangChain Expression Language (LCEL)
chain = prompt_template | model | StrOutputParser()
# chain = prompt_template | model | StrOutputParser()

# Run the chain
result = chain.invoke({"topic": "lawyers", "joke_count": 3})

# Output
print(result)

1. **The Lawyer’s GPS**  
   “I told my GPS, ‘Take me to the nearest courthouse.’ It said, ‘Sure thing. Just follow the trail of lawsuits.’”

2. **The Lawyer’s Diet**  
   “I asked a lawyer how he keeps his diet in check. He said, ‘I only eat cases that are well‑prepared.’”

3. **The Lawyer’s Alarm Clock**  
   “Why don’t lawyers need an alarm clock? Because their deadlines always wake them up… with a *bail* of a call.”


## Output Parser

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(" Return a JSON object with an answer key that answe the following question: {question}")

json_parser = SimpleJsonOutputParser()

json_chains= json_prompt | model | json_parser

In [3]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [4]:
res = json_chains.invoke({"question": "What is the capital of India?"})

In [5]:
res

{'answer': 'New Delhi'}

In [6]:
type(res)

dict

Optionally, you can use Pydantic to define a custom output format

In [7]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

In [8]:
class Joke(BaseModel):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")

In [9]:
parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template="Answer the user query \n {format_instruction} \n {query} \n",
    input_variables=["query"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

chains = prompt | model | parser

In [10]:
res = chains.invoke({"query": "Tell me a joke"})

In [11]:
res

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}